该代码围绕「因果推断中的 Uplift Modeling」展开，核心目标是评估 “优惠干预”（如买一送一）对客户转化的影响，并找到 “干预效果最显著的客户群体”

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from lightgbm import LGBMRegressor

# 设置随机种子，保证结果可复现
np.random.seed(123)
plt.rcParams['font.sans-serif'] = ['SimHei']  # 解决中文显示问题
plt.rcParams['axes.unicode_minus'] = False    # 解决负号显示问题
 
# -------------------------- 2. 传统因果推断模型（T-Learner & X-Learner） --------------------------
# 注：原代码未定义train/test数据集，此处假设已通过train_test_split划分
# 且T、X、y为预定义变量：T=干预变量（0=对照，1=干预），X=特征，y=目标（转化）

# -------------------------- 2.1 T-Learner（最基础的异质性处理效应模型） --------------------------
# 核心逻辑：分别为干预组和对照组训练模型，CATE = 干预组预测 - 对照组预测
# 步骤1：初始化并训练干预组/对照组模型（用LGBMRegressor，因目标是转化概率预测）
m0 = LGBMRegressor(max_depth=2, min_child_samples=60)  # 对照组模型（T=0）
m1 = LGBMRegressor(max_depth=2, min_child_samples=60)  # 干预组模型（T=1）

# 步骤2：分样本训练（仅用对照组数据训m0，仅用干预组数据训m1）
m0.fit(train.query(f"{T}==0")[X], train.query(f"{T}==0")[y])
m1.fit(train.query(f"{T}==1")[X], train.query(f"{T}==1")[y])


 
# 步骤3：估计CATE（个体处理效应）
t_learner_cate_train = m1.predict(train[X]) - m0.predict(train[X])  # 训练集CATE
t_learner_cate_test = test.assign(
    cate=m1.predict(test[X]) - m0.predict(test[X])  # 测试集CATE（新增cate列）
)

# 步骤4：可视化累计增益曲线（评估模型排序能力：高CATE客户是否真的有高转化提升）
gain_curve_test = cumulative_gain(t_learner_cate_test, "cate", y="converted", t="em1")
gain_curve_train = cumulative_gain(
    train.assign(cate=t_learner_cate_train),
    "cate", y="converted", t="em1"
)

plt.plot(gain_curve_test, color="C0", label="测试集")
plt.plot(gain_curve_train, color="C1", label="训练集")
# 基准线：随机选择客户的累计增益（ATE×样本占比）
plt.plot([0, 100], [0, elast(test, "converted", "em1")],
         linestyle="--", color="black", label="基准线（随机）")
plt.legend()
plt.title("T-Learner 累计增益曲线")
plt.xlabel("客户占比（%）")
plt.ylabel("累计转化提升")
plt.show()


NameError: name 'train' is not defined

In [3]:
from sklearn.linear_model import LogisticRegression
 
np.random.seed(123)
 
# first stage models
m0 = LGBMRegressor(max_depth=2, min_child_samples=30)
m1 = LGBMRegressor(max_depth=2, min_child_samples=30)
 
# propensity score model
g = LogisticRegression(solver="lbfgs", penalty='none') 
 
m0.fit(train.query(f"{T}==0")[X], train.query(f"{T}==0")[y])
m1.fit(train.query(f"{T}==1")[X], train.query(f"{T}==1")[y])
                       
g.fit(train[X], train[T]);

NameError: name 'LGBMRegressor' is not defined

In [ ]:
d_train = np.where(train[T]==0,
                   m1.predict(train[X]) - train[y],
                   train[y] - m0.predict(train[X]))
 
# second stage
mx0 = LGBMRegressor(max_depth=2, min_child_samples=30)
mx1 = LGBMRegressor(max_depth=2, min_child_samples=30)
 
mx0.fit(train.query(f"{T}==0")[X], d_train[train[T]==0])
mx1.fit(train.query(f"{T}==1")[X], d_train[train[T]==1]);

In [ ]:
def ps_predict(df, t): 
    return g.predict_proba(df[X])[:, t]
    
    
x_cate_train = (ps_predict(train,1)*mx0.predict(train[X]) +
                ps_predict(train,0)*mx1.predict(train[X]))
 
x_cate_test = test.assign(cate=(ps_predict(test,1)*mx0.predict(test[X]) +
                                ps_predict(test,0)*mx1.predict(test[X])))

In [ ]:
gain_curve_test = cumulative_gain(x_cate_test, "cate", y="converted", t="em1")
gain_curve_train = cumulative_gain(train.assign(cate=x_cate_train), "cate", y="converted", t="em1")
plt.plot(gain_curve_test, color="C0", label="Test")
plt.plot(gain_curve_train, color="C1", label="Train")
plt.plot([0, 100], [0, elast(test, "converted", "em1")], linestyle="--", color="black", label="Baseline")
plt.legend();
plt.title("X-Learner");

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import xgboost as xgb
from sklearn.model_selection import train_test_split
 
import datetime
 
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, classification_report, roc_auc_score
from sklearn import metrics
from xgboost.sklearn import XGBClassifier
from xgboost import XGBClassifier
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
import lightgbm
import graphviz
from sklearn import tree 
from sklearn.tree import DecisionTreeClassifier
 
# 1. 读入数据
# 该数据集 来源  https://www.kaggle.com/datasets/davinwijaya/customer-retention
data = pd.read_csv('./data.csv' )
data.head()
 
 
# 2.查看离散变量及取值
for column in data.drop(columns = ['recency', 'history']):
    print(column, '\n', data[column].value_counts(), '\n')
 
# 3.连续变量看分布
con_columns = ['recency', 'history']
 
fig = plt.figure(figsize = (16, 10))
ax1 = fig.add_subplot(121)
data['recency'].hist(ax = ax1)
ax1.set_title('recency distribution')
 
ax2 = fig.add_subplot(122)
data['history'].hist(ax = ax2)
ax2.set_title('history distribution')
 
 
# 4.目标变量及干预变量列名修改
df_model = data.rename(columns = {'conversion': 'target'})  # 目标变量
df_model = df_model.rename(columns = {'offer': 'treatment'})  # 是否干预
 
df_model.treatment = df_model.treatment.map({'No Offer': 0, 'Buy One Get One': -1, 'Discount': 1})  # treatment重新赋值
 
# 离散变量转为哑变量
df_model = pd.get_dummies(df_model)
df_model.head()
 
# 看不同干预条件下的转化情况
df_model.groupby('treatment').agg({'target': 'mean'})
 
 
 
# 5.定义方法：看各特征下转化率的提升
def get_every_group_up(data, dim, ab_tag):
    # 按输入维度 dim、ab分组字段 ab_tag 进行分组汇总件量和空运件量
    dim_data = data.groupby(by = [dim, ab_tag]).agg({'target': 'sum', 'treatment': 'count'})
    dim_data.columns = ['target', 'cnt']  # 将treatment 列名 赋值为 cnt 用户数
    dim_data['pct'] = dim_data['target']/dim_data['cnt']  # 空运件量占比
    
 
    # 计算各因素实验组和对照组的空运寄件率
    dim_data_b = dim_data.loc[(slice(None), -1), 'pct'].loc[:, -1]
    dim_data_a = dim_data.loc[(slice(None), 0), 'pct'].loc[:, 0]
 
    # 计算各因素分布情况
    dim_data_pct = data[dim].value_counts()/data[dim].count()
 
    dim_data_df = pd.concat([dim_data_pct, dim_data_a, dim_data_b], axis = 1)
    
    # 列重命名 占比 对照组转化率 实验组转化率
    dim_data_df.columns = ['proportion', 'pct_a', 'pct_b']
    dim_data_df['uplift'] = dim_data_df['pct_b'] - dim_data_df['pct_a']
    
    return dim_data_df
 
 
# 查看 recency 不同分段下 实验组（买一送一）与对照组（无优惠）转化率提升情况
uplift_data = df_model[df_model['treatment'].isin([-1, 0])]
 
uplift_data['recency_bins'], updown = pd.qcut(uplift_data['recency'], q = 5, retbins = True, duplicates = 'drop')
diff_recency_bins_df = get_every_group_up(uplift_data, 'recency_bins', 'treatment')
diff_recency_bins_df.style.background_gradient()

字段说明:
recency:距离上次购买的月数
history:历史购买金额
used discount: 用户之前购买时是否使用折扣
used bogo:买一送一，
zip code:地区等级，uburban 郊区 /Urban 城市 /Rural 多村 其他
is referral: 客户是否通过推荐渠道获得
channel:客户使用渠道ofer: 发送给用户的优惠类型，折扣/买一送一/不优惠 这个字段可以理解为Treatment,可以分别看 折扣较不优惠的购买提升情况conversion 客户是否转化


In [ ]:
# 定义方法 画模型roc曲线 阈值等，模型调参时查看效果较方便
def plot_model_result(preds, y_test):
    # roc曲线相关指标
    FPR, recall, thresholds = roc_curve(y_test, preds, pos_label = 1)
    area = roc_auc_score(y_test, preds)
    
    plt.figure()
    plt.plot(FPR, recall, color = 'r', label = 'roc curve (area=%0.4f' % area)
    plt.plot([0, 1], [0, 1], color = 'black', linestyle = '--')
    maxindex = (recall - FPR).tolist().index(max(recall - FPR))
    plt.scatter(FPR[maxindex], recall[maxindex], c = 'black', s = 30)
    print('阈值', thresholds[maxindex])
    plt.xlim([-0.05, 1.05])
    plt.ylim([-0.05, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('Recall')
    plt.title('ROC Curve')
    plt.legend(loc = 'best')
    plt.show()
    
    ypred = preds.copy() 
    ypred[preds > thresholds[maxindex]] = 1 
    ypred[ypred != 1] = 0
    print('classification_report \n',classification_report(y_test, ypred, digits=4))
#     print('confusion_matrix \n', confusion_matrix(y_test, ypred))
#     return thresholds[maxindex]  # 将阈值返回

In [ ]:
# 1. 模型数据准备 以其中 Buy One Get One 即 treatment = -1 为例
# 对照组 建模
uplift_data_a = uplift_data[uplift_data['treatment'] == 0].drop(columns = ['recency_bins', 'history_bins'])
 
# 实验组
uplift_data_b = uplift_data[uplift_data['treatment'] == -1].drop(columns = ['recency_bins', 'history_bins'])
 
x_a = uplift_data_a.loc[:, uplift_data_a.columns.difference(['treatment', 'target'])]
y_a = uplift_data_a['target']
 
 
# 划分训练集与测试集
x_train_a, x_test_a, y_train_a, y_test_a = train_test_split(x_a, y_a, test_size = 0.4, random_state = 20000)
 
# 建模 这里以 RandomForest 为例
clf_rf_a = RandomForestClassifier(n_estimators = 100, random_state = 200, max_depth = 10, min_samples_leaf = 10)
clf_rf_a.fit(x_train_a, y_train_a)
 
# 在测试集上预测并看效果
y_pre_test_a = clf_rf_a.predict_proba(x_test_a)
plot_model_result(y_pre_test_a[:, 1], y_test_a)
 
 
 
# 实验组建模 和 对照组建模类似
# lgb 模型数据集 实验组
x_b = uplift_data_b.loc[:, uplift_data_a.columns.difference(['treatment', 'target'])]
y_b = uplift_data_b['target']
 
x_train_b, x_test_b, y_train_b, y_test_b = train_test_split(x_b, y_b, test_size = 0.4, random_state = 20000)
 
clf_rf_b = RandomForestClassifier(n_estimators = 100, random_state = 200, max_depth = 10, min_samples_leaf = 10)
clf_rf_b.fit(x_train_b, y_train_b)
 
y_pre_test_b = clf_rf_b.predict_proba(x_test_b)
plot_model_result(y_pre_test_b[:, 1], y_test_b)

In [ ]:
# 1.测试集上的实验组、对照组分别拿两个模型预测
# pre_a 是 对照组用户x_valid_a  在没有干预情况下的寄件率，用实验组 模型 model_b 得到在有干预的情况下的空运寄件率
pre_a_under_b_model = clf_rf_b.predict_proba(x_test_a)
 
 
# pre_b 是 实验组用户 x_valid_b 在有干预情况下的寄件率，用对照组 模型 model_a 得到在没有干预情况下的空运寄件率
pre_b_under_a_model = clf_rf_a.predict_proba(x_test_b)
 
 
# 2.实验组、对照组数据集分别添加以上两预测值
# 对照组：将两种情况下的寄件率拼接到 对应的数据集
x_test_a['pre_a'] = y_pre_test_a[:,0]
x_test_a['pre_b'] = pre_a_under_b_model[:,0]
x_test_a.head()
 
test_a = pd.concat([x_test_a, y_test_a], axis = 1)
test_a.head()
 
 
# 实验组：将两种情况下的寄件率拼接到 对应的数据集上,
x_test_b['pre_a'] = y_pre_test_b[:,0]
x_test_b['pre_b'] = pre_b_under_a_model[:,0]
x_test_b.head()
 
test_b = pd.concat([x_test_b, y_test_b], axis = 1)
test_b.head()
 
 
# 3.计算预测出的uplift score
test_a['uplift_score'] = test_a['pre_b'] - test_a['pre_a']
test_b['uplift_score'] = test_b['pre_b'] - test_b['pre_a']
 
# 4.合并两组数据，一起划分十分位
test_ab_all = pd.concat([test_a, test_b])
test_ab_all = pd.concat([test_ab_all, data[['channel', 'zip_code', 'offer']]], axis = 1 , join = 'inner')
test_ab_all['treatment'] = test_ab_all['offer'].map(lambda x: 0 if x == 'No Offer' else 1)
test_ab_all['uplift_bins'], updown = pd.qcut(test_ab_all['uplift_score'], q = 10, retbins = True, duplicates = 'drop', labels = np.arange(1, 11))
test_ab_all
 
# 5.计算每个十分位上，实验组、对照组的真实转化率 c_pct = target求和 / 十分位内各自记录数
test_ab_all_group = test_ab_all.groupby(by = ['uplift_bins', 'treatment']).agg({'history': 'count', 'target': 'sum', 'pre_b': 'mean', 'pre_a': 'mean', 'uplift_score': 'mean'})
test_ab_all_group['c_pct'] = test_ab_all_group['target']/test_ab_all_group['history']
 
 
test_ab_all_group.columns = ['user_num', 'c_user_num', 'pre_b_mean', 'pre_a_mean', 'uplift_score_mean', 'c_pct']
test_ab_all_group
 
# 6.从以上数据中分离实验组、对照组数据
b_group = test_ab_all_group.loc[(slice(None), [1]),:]
a_group = test_ab_all_group.loc[(slice(None), [0]),:]
 
# multiindex处理方法
a_group_ = a_group.xs(key = 0, level = 1)
b_group_ = b_group.xs(key = 1, level = 1)
 
# 7.将以上十分位内的对照组数据拼接到实验组中，以计算每十分位内的真实增益
b_group_ = pd.concat([b_group_, a_group_['c_pct']], axis = 1)
b_group_.columns = ['user_num', 'c_user_num', 'pre_b_mean', 'pre_a_mean', 'uplift_score_mean', 'b_c_pct', 'a_c_pct']
 
# real_diff_pct 为 真实增益，拉齐后的实验组转化率 - 对照组转化率
b_group_['real_diff_pct'] = b_group_['b_c_pct'] - b_group_['a_c_pct'] 
b_group_.index = [10, 9, 8, 7, 6, 5, 4, 3, 2, 1]
b_group_
 
plt.bar(b_group_.index, b_group_['real_diff_pct'])

In [ ]:
# 1. 分别划分十分位
test_a['uplift_bins'], updown = pd.qcut(test_a['uplift_score'], q = 10, retbins = True, duplicates = 'drop', labels = np.arange(1, 11))
 
test_b['uplift_bins'], updown = pd.qcut(test_b['uplift_score'], q = 10, retbins = True, duplicates = 'drop', labels = np.arange(1, 11))
 
 
# 2. 以uplift十分位进行聚合，计算每个十分位中的用户数、真实转化数、转化率
test_a_group = test_a.groupby(by = ['uplift_bins']).agg({'history': 'count', 'target': 'sum'})
test_a_group.columns = ['user_num', 'c_user_num']
 
test_a_group['kw_pct'] = test_a_group['c_user_num']/test_a_group['user_num']
test_a_group
 
test_b_group = test_b.groupby(by = ['uplift_bins']).agg({'history': 'count', 'target': 'sum'})
test_b_group.columns = ['user_num', 'c_user_num']
 
test_b_group['kw_pct'] = test_b_group['c_user_num']/test_a_group['user_num']
test_b_group
 
 
# 3. 合并实验组、对照组数据，并重命名列名
test_union_group = pd.concat([test_b_group, test_a_group], axis = 1)
test_union_group.columns = ['user_num_b', 'c_user_num_b', 'c_pct_b', 'user_num_a', 'c_user_num_a', 'c_pct_a']
test_union_group['uplift_score'] = test_union_group['c_pct_b'] - test_union_group['c_pct_a']
test_union_group
 
test_union_group.index = [10, 9, 8, 7, 6, 5, 4, 3, 2, 1]
test_union_group
 
 
# 5. 添加实验组、对照组 总的用户数
test_union_group['total_b'] = test_union_group['user_num_b'].sum()
test_union_group['total_a'] = test_union_group['user_num_a'].sum()
 
# 6. 分别计算实验组、对照组 累计转化用户数
test_union_group['c_user_num_b_cums'] = test_union_group['c_user_num_b'].cumsum(axis = 0)
test_union_group['c_user_num_a_cums'] = test_union_group['c_user_num_a'].cumsum(axis = 0)
 
# 7. 计算累计的转化率之差
test_union_group['q_b_i'] = test_union_group['c_user_num_b_cums']/test_union_group['total_b']
 
test_union_group['q_a_i'] = test_union_group['c_user_num_a_cums']/test_union_group['total_a']
 
test_union_group['q_i'] = test_union_group['q_b_i'] - test_union_group['q_a_i']
 
 
# 8. 引入包画gini曲线，并计算AUUC
from numpy import trapz  # 利用梯度规则求解积分
from scipy.integrate import simps  # 利用辛普森积分法（Simpson's rule），以二次曲线逼近的方式取代矩形或梯形积分公式，以求得定积分的数值近似解。
 
x = np.r_[0, list(test_union_group.index)]
y = np.r_[0, list(test_union_group['q_i'])]
 
random_curve_area = simps([0, y[len(y) - 1]], [0,10], dx=0.001)  # 随机曲线与x轴围成的面积
gini_curve_area = simps(y, x, dx=0.001)   # gini曲线与x轴围成的面积
auuc = gini_curve_area - random_curve_area
 
fig, ax = plt.subplots()
ax.plot(x, y, c = 'green', label = 'gini curve')
ax.plot([0,10], [0, y[len(y) - 1]], c = 'blue', ls = '--', label = 'random curve')
ax.text(x = 7.5, y = 0.02, s = 'auuc: %f'%auuc )